# 1.1 Ingest PAMAP2 into S3 raw and curated (Parquet) zones

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import boto3
from src.pamap2.config import load_config, s3_uri
cfg = load_config()
session = boto3.Session(region_name=cfg["region"])
sts = session.client("sts")
print("Caller:", sts.get_caller_identity()["Arn"])
import zipfile, shutil, requests, pandas as pd
from botocore.exceptions import ClientError
from src.pamap2 import schema
s3 = session.client("s3")
bucket = cfg["bucket"]
SCRATCH = pathlib.Path("/tmp/pamap2")
SCRATCH.mkdir(parents=True, exist_ok=True)

In [ ]:
URL = "https://archive.ics.uci.edu/static/public/231/pamap2+physical+activity+monitoring.zip"
zpath = SCRATCH / "pamap2.zip"
if not zpath.exists():
    with requests.get(URL, stream=True, timeout=120) as r:
        r.raise_for_status()
        part = zpath.with_name(zpath.name + ".part")
        with open(part, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
    part.replace(zpath)
def extract(zp, dest):
    with zipfile.ZipFile(zp) as z:
        z.extractall(dest)
extract(zpath, SCRATCH / "unzipped")
for inner in list((SCRATCH / "unzipped").rglob("*.zip")):   # UCI zip nests PAMAP2_Dataset.zip
    extract(inner, SCRATCH / "unzipped")
dats = sorted((SCRATCH / "unzipped").rglob("subject*.dat"))
print(len(dats), "files:", [d.parent.name + "/" + d.name for d in dats])

In [ ]:
def s3_size(key):
    try:
        return s3.head_object(Bucket=bucket, Key=key)["ContentLength"]
    except ClientError as e:
        if e.response["Error"]["Code"] not in ("404", "NoSuchKey", "NotFound"):
            raise
        return None

total = 0
for d in dats:
    session_name = d.parent.name.lower()           # protocol | optional
    subject_id = int(d.stem.replace("subject", ""))
    raw_key = f"{cfg['s3_prefixes']['raw']}pamap2/{session_name}/{d.name}"
    if s3_size(raw_key) == d.stat().st_size:
        print("raw exists, skipping:", raw_key)
    else:
        s3.upload_file(str(d), bucket, raw_key)      # byte-for-byte unchanged
        print("uploaded", raw_key)
    df = schema.parse_dat(d, subject_id=subject_id, session=session_name)
    out = SCRATCH / f"{session_name}_{subject_id}.parquet"
    df.drop(columns=["subject_id", "session"]).to_parquet(out, index=False)
    cur_key = (f"{cfg['s3_prefixes']['curated']}pamap2/session={session_name}/"
               f"subject_id={subject_id}/part-0.parquet")
    s3.upload_file(str(out), bucket, cur_key)         # overwrite is idempotent
    total += len(df)
    print(f"curated {session_name}/{subject_id}: {len(df)} rows")
print("Total parsed rows:", total)